In [15]:
import boto3
import json
import pandas as pd

s3 = boto3.client("s3")
BUCKET = "yankees-pipeline-andre"

# grab one file to start
key = "raw/game_logs/2026-06-02/player_680474_hitting.json"
obj = s3.get_object(Bucket=BUCKET, Key=key)
data = json.loads(obj["Body"].read())

In [16]:
split = data['stats'][0]['splits']
df = pd.json_normalize(split)
df.columns

Index(['season', 'date', 'gameType', 'isHome', 'isWin', 'positionsPlayed',
       'stat.summary', 'stat.gamesPlayed', 'stat.flyOuts', 'stat.groundOuts',
       'stat.airOuts', 'stat.runs', 'stat.doubles', 'stat.triples',
       'stat.homeRuns', 'stat.strikeOuts', 'stat.baseOnBalls',
       'stat.intentionalWalks', 'stat.hits', 'stat.hitByPitch', 'stat.avg',
       'stat.atBats', 'stat.obp', 'stat.slg', 'stat.ops',
       'stat.caughtStealing', 'stat.stolenBases', 'stat.stolenBasePercentage',
       'stat.caughtStealingPercentage', 'stat.groundIntoDoublePlay',
       'stat.groundIntoTriplePlay', 'stat.numberOfPitches',
       'stat.plateAppearances', 'stat.totalBases', 'stat.rbi',
       'stat.leftOnBase', 'stat.sacBunts', 'stat.sacFlies', 'stat.babip',
       'stat.groundOutsToAirouts', 'stat.catchersInterference',
       'stat.atBatsPerHomeRun', 'team.id', 'team.name', 'team.link',
       'player.id', 'player.fullName', 'player.link', 'league.id',
       'league.name', 'league.link', 

In [27]:
pd.set_option('display.max_columns', None)
df.head(1)


,season,date,gameType,isHome,isWin,positionsPlayed,stat.summary,stat.gamesPlayed,stat.flyOuts,stat.groundOuts,stat.airOuts,stat.runs,stat.doubles,stat.triples,stat.homeRuns,stat.strikeOuts,stat.baseOnBalls,stat.intentionalWalks,stat.hits,stat.hitByPitch,stat.avg,stat.atBats,stat.obp,stat.slg,stat.ops,stat.caughtStealing,stat.stolenBases,stat.stolenBasePercentage,stat.caughtStealingPercentage,stat.groundIntoDoublePlay,stat.groundIntoTriplePlay,stat.numberOfPitches,stat.plateAppearances,stat.totalBases,stat.rbi,stat.leftOnBase,stat.sacBunts,stat.sacFlies,stat.babip,stat.groundOutsToAirouts,stat.catchersInterference,stat.atBatsPerHomeRun,team.id,team.name,team.link,player.id,player.fullName,player.link,league.id,league.name,league.link,sport.id,sport.link,sport.abbreviation,opponent.id,opponent.name,opponent.link,game.gamePk,game.link,game.content.link,game.gameNumber,game.dayNight
0,2026,2026-04-29,R,False,False,"[{'code': '7', 'name': 'Outfielder', 'type': '...",0-1 | BB,1,0,0,1,0,0,0,0,0,1,0,0,0,.000,1,.500,.000,.500,0,0,.---,.---,0,0,13,2,0,0,0,0,0,.000,0.00,0,-.--,147,New York Yankees,/api/v1/teams/147,680474,Max Schuemann,/api/v1/people/680474,103,American League,/api/v1/league/103,1,/api/v1/sports/1,MLB,140,Texas Rangers,/api/v1/teams/140,822907,/api/v1.1/game/822907/feed/live,/api/v1/game/822907/content,1,day


In [26]:
df['positionsPlayed'].apply(len).min()


np.int64(1)

In [34]:
cols = [
    # identity / keys
    'player.id', 'player.fullName', 'game.gamePk', 'date', 'season',
    # dimensions
    'sport.id', 'sport.abbreviation', 'league.id', 'league.name',
    'team.id', 'team.name', 'opponent.id', 'opponent.name',
    'gameType', 'isHome', 'game.dayNight', 'positionsPlayed',
    # stats
    'stat.runs', 'stat.hits', 'stat.doubles', 'stat.triples', 'stat.homeRuns',
    'stat.rbi', 'stat.totalBases', 'stat.baseOnBalls', 'stat.intentionalWalks',
    'stat.strikeOuts', 'stat.hitByPitch', 'stat.atBats', 'stat.plateAppearances',
    'stat.stolenBases', 'stat.caughtStealing', 'stat.groundIntoDoublePlay',
    'stat.numberOfPitches', 'stat.leftOnBase', 'stat.sacBunts', 'stat.sacFlies',
]

df = df[cols]
df.head()

,player.id,player.fullName,game.gamePk,date,season,sport.id,sport.abbreviation,league.id,league.name,team.id,team.name,opponent.id,opponent.name,gameType,isHome,game.dayNight,positionsPlayed,stat.runs,stat.hits,stat.doubles,stat.triples,stat.homeRuns,stat.rbi,stat.totalBases,stat.baseOnBalls,stat.intentionalWalks,stat.strikeOuts,stat.hitByPitch,stat.atBats,stat.plateAppearances,stat.stolenBases,stat.caughtStealing,stat.groundIntoDoublePlay,stat.numberOfPitches,stat.leftOnBase,stat.sacBunts,stat.sacFlies
0,680474,Max Schuemann,822907,2026-04-29,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,False,day,"[{'code': '7', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,1,0,0,0,1,2,0,0,0,13,0,0,0
1,680474,Max Schuemann,823555,2026-05-03,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,"[{'code': '9', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,680474,Max Schuemann,823552,2026-05-04,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,"[{'code': '9', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,680474,Max Schuemann,823551,2026-05-07,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,True,day,"[{'code': '6', 'name': 'Shortstop', 'type': 'I...",0,1,1,0,0,1,2,0,0,1,0,4,4,0,0,0,16,4,0,0
4,680474,Max Schuemann,823792,2026-05-09,2026,1,MLB,103,American League,147,New York Yankees,158,Milwaukee Brewers,R,False,day,"[{'code': '12', 'name': 'Pinch Runner', 'type'...",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [35]:
mapped_cols = {
    'player.id': 'player_id',
    'player.fullName': 'player_name',
    'game.gamePk': 'game_pk',
    'date': 'game_date',
    'sport.id': 'sport_id',
    'sport.abbreviation': 'level',
    'league.id': 'league_id',
    'league.name': 'league_name',
    'team.id': 'team_id',
    'team.name': 'team_name',
    'opponent.id': 'opponent_id',
    'opponent.name': 'opponent_name',
    'gameType': 'game_type',
    'isHome': 'is_home',
    'game.dayNight': 'day_night',
    'positionsPlayed': 'positions_played',
    'stat.runs': 'r',
    'stat.hits': 'h',
    'stat.doubles': 'x2b',
    'stat.triples': 'x3b',
    'stat.homeRuns': 'hr',
    'stat.rbi': 'rbi',
    'stat.totalBases': 'tb',
    'stat.baseOnBalls': 'bb',
    'stat.intentionalWalks': 'ibb',
    'stat.strikeOuts': 'so',
    'stat.hitByPitch': 'hbp',
    'stat.atBats': 'ab',
    'stat.plateAppearances': 'pa',
    'stat.stolenBases': 'sb',
    'stat.caughtStealing': 'cs',
    'stat.groundIntoDoublePlay': 'gidp',
    'stat.numberOfPitches': 'np',
    'stat.leftOnBase': 'lob',
    'stat.sacBunts': 'sh',
    'stat.sacFlies': 'sf',
}

df = df.rename(columns=mapped_cols)
df.head()

,player_id,player_name,game_pk,game_date,season,sport_id,level,league_id,league_name,team_id,team_name,opponent_id,opponent_name,game_type,is_home,day_night,positions_played,r,h,x2b,x3b,hr,rbi,tb,bb,ibb,so,hbp,ab,pa,sb,cs,gidp,np,lob,sh,sf
0,680474,Max Schuemann,822907,2026-04-29,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,False,day,"[{'code': '7', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,1,0,0,0,1,2,0,0,0,13,0,0,0
1,680474,Max Schuemann,823555,2026-05-03,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,"[{'code': '9', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,680474,Max Schuemann,823552,2026-05-04,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,"[{'code': '9', 'name': 'Outfielder', 'type': '...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,680474,Max Schuemann,823551,2026-05-07,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,True,day,"[{'code': '6', 'name': 'Shortstop', 'type': 'I...",0,1,1,0,0,1,2,0,0,1,0,4,4,0,0,0,16,4,0,0
4,680474,Max Schuemann,823792,2026-05-09,2026,1,MLB,103,American League,147,New York Yankees,158,Milwaukee Brewers,R,False,day,"[{'code': '12', 'name': 'Pinch Runner', 'type'...",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
df['positions_played'] = df['positions_played'].apply(lambda lst: ','.join(p['abbreviation'] for p in lst) if lst else '')
df['positions_played'][0]

0        LF
1        RF
2        RF
3     SS,RF
4     PR,LF
5        SS
6        SS
7     PR,LF
8     LF,3B
9        PR
10    PR,LF
11    PR,RF
12    2B,RF
Name: positions_played, dtype: str

In [50]:
df['game_date'] = pd.to_datetime(df['game_date'], format='%Y-%m-%d')
df['game_date'].dtype

dtype('<M8[us]')

In [49]:
df.head()

,player_id,player_name,game_pk,game_date,season,sport_id,level,league_id,league_name,team_id,team_name,opponent_id,opponent_name,game_type,is_home,day_night,positions_played,r,h,x2b,x3b,hr,rbi,tb,bb,ibb,so,hbp,ab,pa,sb,cs,gidp,np,lob,sh,sf
0,680474,Max Schuemann,822907,2026-04-29,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,False,day,LF,0,0,0,0,0,0,0,1,0,0,0,1,2,0,0,0,13,0,0,0
1,680474,Max Schuemann,823555,2026-05-03,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,RF,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,680474,Max Schuemann,823552,2026-05-04,2026,1,MLB,103,American League,147,New York Yankees,110,Baltimore Orioles,R,True,day,RF,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,680474,Max Schuemann,823551,2026-05-07,2026,1,MLB,103,American League,147,New York Yankees,140,Texas Rangers,R,True,day,"SS,RF",0,1,1,0,0,1,2,0,0,1,0,4,4,0,0,0,16,4,0,0
4,680474,Max Schuemann,823792,2026-05-09,2026,1,MLB,103,American League,147,New York Yankees,158,Milwaukee Brewers,R,False,day,"PR,LF",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [53]:
df.dtypes

player_id                    int64
player_name                    str
game_pk                      int64
game_date           datetime64[us]
season                       int64
sport_id                     int64
level                          str
league_id                    int64
league_name                    str
team_id                      int64
team_name                      str
opponent_id                  int64
opponent_name                  str
game_type                      str
is_home                       bool
day_night                      str
positions_played               str
r                            int64
h                            int64
x2b                          int64
x3b                          int64
hr                           int64
rbi                          int64
tb                           int64
bb                           int64
ibb                          int64
so                           int64
hbp                          int64
ab                  

In [52]:
df['season'] = df['season'].astype(int)